# Logistic Regression – Telco Customer Churn

**Name:** RENDLA PAVITHRA  
**ID:** 2300090355

### Objective

The aim of this analysis is to predict whether a telecom customer is likely to churn or stay.

The model will use customer information such as tenure, contract type, services and monthly charges. At the end, the model will be checked using suitable classification metrics.

## 1. Import libraries

I am using pandas for the data, matplotlib and seaborn for simple graphs, and scikit-learn for the model and evaluation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

## 2. Load the dataset

The dataset contains customer details, services used, charges and the churn result.

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

## 3. Check the data

I checked the columns, number of rows and missing values before starting the model.

In [ ]:
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

## 4. Basic analysis

The churn column tells us whether a customer left the company. I also checked the number of customers in each group.

In [ ]:
print("Churn count:")
print(df["Churn"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

## 5. Prepare the data

`TotalCharges` is converted to numeric because some values are blank in the original dataset. The customer ID is not used because it is only an identifier.

I also converted the churn values into 0 and 1.

In [ ]:
data = df.copy()

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["SeniorCitizen"] = data["SeniorCitizen"].astype(str)

data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

print("Churn rate:", round(data["Churn"].mean() * 100, 2), "%")

## 6. Select the inputs and output

I selected customer details, services, contract information, tenure and charges as the input variables.

`Churn` is the value that the model needs to predict.

In [ ]:
features = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges"
]

X = data[features]
y = data["Churn"]

print("Input columns:", X.columns.tolist())
print("Target column:", y.name)

## 7. Split the data

I used 80% of the data for training and 20% for testing. The test data is kept separate so the model can be checked on customers it did not train on.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 8. Prepare the columns

Logistic Regression needs numerical values. Numeric columns are scaled and categorical columns are converted using one-hot encoding.

In [ ]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [col for col in features if col not in numeric_features]

numeric_process = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_process = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_process, numeric_features),
    ("categorical", categorical_process, categorical_features)
])

## 9. Train the model

I used Logistic Regression because it is simple and gives a probability for each customer.

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


## 10. Predict customer churn

The model gives a probability of churn. I used 0.50 as the first cut-off.

If the probability is 0.50 or more, the customer is classified as likely to churn.

In [ ]:
probability = model.predict_proba(X_test)[:, 1]
prediction = (probability >= 0.50).astype(int)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Churn Probability": probability.round(3),
    "Prediction": prediction
})

results["Prediction"] = results["Prediction"].map({
    0: "Likely to stay",
    1: "Likely to churn"
})

results.head(10)

## 11. Check model performance

I used Accuracy, Precision, Recall, F1-score and ROC-AUC to check the model.

Recall is especially important here because the company wants to identify customers who may leave.

In [ ]:
accuracy = accuracy_score(y_test, prediction)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
auc = roc_auc_score(y_test, probability)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")

Accuracy  : 0.7374
Precision : 0.5034
Recall    : 0.7834
F1 Score  : 0.6129
ROC-AUC   : 0.8400


## 12. Confusion matrix

The confusion matrix shows how many churn and non-churn customers were classified correctly or incorrectly.

In [ ]:
cm = confusion_matrix(y_test, prediction)

tn, fp, fn, tp = cm.ravel()

print("Correctly identified churn customers:", tp)
print("Non-churn customers incorrectly flagged:", fp)
print("Churn customers missed:", fn)
print("Correctly identified non-churn customers:", tn)

print("\nClassification Report:")
print(classification_report(
    y_test,
    prediction,
    target_names=["Likely to stay", "Likely to churn"],
    digits=3
))

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Stay", "Churn"],
    yticklabels=["Stay", "Churn"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

Correctly identified churn customers: 293
Non-churn customers incorrectly flagged: 289
Churn customers missed: 81
Correctly identified non-churn customers: 746

Classification Report:
                 precision    recall  f1-score   support

Likely to stay      0.902     0.721     0.801      1035
Likely to churn     0.503     0.783     0.613       374

accuracy                                0.737      1409
macro avg           0.703     0.752     0.707      1409
weighted avg        0.796     0.737     0.751      1409


## 13. Predict new customer records

The same trained model can be used for new customer details. Below, a few test records are shown with their churn probability and prediction.

In [ ]:
new_customers = X_test.sample(5, random_state=7).copy()

new_probability = model.predict_proba(new_customers)[:, 1]

new_results = new_customers[
    ["tenure", "Contract", "InternetService", "MonthlyCharges", "TotalCharges"]
].copy()

new_results["Churn Probability"] = new_probability.round(3)
new_results["Prediction"] = [
    "Likely to churn" if p >= 0.50 else "Likely to stay"
    for p in new_probability
]

new_results.reset_index(drop=True)

## 14. Final findings

The model correctly identified **293 customers who churned** and missed **81 churn customers**.

It also flagged **289 customers as churn risk even though they did not churn**.

For this problem, missing a customer who is actually going to leave can be more costly than giving an offer to a customer who would have stayed. So, in a real company, the probability cut-off can be adjusted to focus more on recall.

### Business recommendation

The company can use the model to identify higher-risk customers and provide retention offers before they leave. A useful next improvement would be to tune the prediction threshold based on the actual cost of retention offers and the value of each customer.

### Dataset

Telco Customer Churn – Kaggle

https://www.kaggle.com/datasets/blastchar/telco-customer-churn